# Demo 1 — What can a bounded OLS association support?

**Learning question:** In the hypothetical participant population represented by this teaching process, what is the coefficient on study hours in the conditional mean model for assessment score after accounting for prior score?

The input grain is one synthetic workshop participant. The output grain is one target-coefficient row and one interval row for a supplied new case. The target **population** is the hypothetical collection described by the teaching process, and the **estimand** is its population coefficient on study hours. The result describes **association**—a conditional relationship—not **causation**, an effect of intervening on study time.

This Colab-first notebook runs equivalently in local Jupyter. Colab files are ephemeral, and edits opened from GitHub are not automatically saved back to GitHub. Assignment use of Colab remains conditional on the repository-save and Classroom 50 pilot. Use only the synthetic, non-identifying fixture; do not add private data or credentials. Restart the kernel and run every cell in order because stored output is not fresh-execution evidence.

Before fitting, predict the sign of the study-hours coefficient and phrase its bounded, prior-score-conditional meaning.

In [ ]:
from importlib import metadata
from pathlib import Path
import platform
import subprocess
import sys

EXPECTED_PYTHON = "3.12.13"
EXPECTED_DISTRIBUTIONS = {
    "numpy": "2.0.2",
    "pandas": "3.0.3",
    "statsmodels": "0.14.6",
    "scikit-learn": "1.9.0",
    "matplotlib": "3.11.1",
}

def distribution_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None

mismatched = [
    f"{name}=={expected}"
    for name, expected in EXPECTED_DISTRIBUTIONS.items()
    if distribution_version(name) != expected
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

actual_versions = {
    name: distribution_version(name) for name in EXPECTED_DISTRIBUTIONS
}
assert platform.python_version() == EXPECTED_PYTHON
assert actual_versions == EXPECTED_DISTRIBUTIONS

starting_directory = Path.cwd().resolve()
search_bases = (starting_directory, *starting_directory.parents)
demo_root = None
for search_base in search_bases:
    for candidate in (search_base, search_base / "10" / "demo"):
        if (candidate / "DEMO_GUIDE.md").is_file() and (
            candidate / ".python-version"
        ).is_file():
            demo_root = candidate
            break
    if demo_root is not None:
        break
DEMO_ROOT = demo_root if demo_root is not None else starting_directory

print(f"Python {platform.python_version()}")
for distribution_name, distribution_version_text in actual_versions.items():
    print(f"{distribution_name} {distribution_version_text}")
print(f"Demo root: {DEMO_ROOT}")

## Fix the inference contract before fitting

A **sample** is the observed subset used for estimation; a population is the larger hypothetical collection to which the estimand refers. An **assumption** is a condition required to connect model calculations to that target. This small sample demonstrates mechanics and cannot establish a real-world population claim.

The conditional mean must be adequately represented by the stated linear form. Observations must be independent, or dependence must be handled by the design. Conventional coefficient intervals assume reasonably stable residual variance and an appropriate small-sample error shape. Explanatory variables cannot be exact linear combinations. The sample and measurements must be relevant to the intended population. The intended claim remains association, not causation.

The row key is `participant_id`. The response is assessment score; the explanatory variables are study hours and prior score.

In [ ]:
import hashlib
import io
import numpy as np
import pandas as pd

WORKSHOP_BYTES = b'participant_id,study_hours,prior_score,assessment_score\np01,1,58,65\np02,2,61,69\np03,2,67,72\np04,3,63,71\np05,4,70,78\np06,4,74,80\np07,5,69,79\np08,6,76,86\np09,6,82,90\np10,7,79,88\np11,8,85,94\np12,9,88,98\n'
WORKSHOP_SHA256 = "eefb5f1023e9b84106f407800fa0db72853b6876d58e61255c346ed2d2d32f05"
fixture_path = DEMO_ROOT / "data" / "workshop_participants.csv"
fixture_bytes = fixture_path.read_bytes() if fixture_path.is_file() else WORKSHOP_BYTES
assert len(fixture_bytes) == 200
assert hashlib.sha256(fixture_bytes).hexdigest() == WORKSHOP_SHA256

workshop_data = pd.read_csv(
    io.BytesIO(fixture_bytes),
    dtype={
        "participant_id": "string",
        "study_hours": "int64",
        "prior_score": "int64",
        "assessment_score": "int64",
    },
)
assert workshop_data.shape == (12, 4)
assert workshop_data.columns.tolist() == [
    "participant_id", "study_hours", "prior_score", "assessment_score"
]
assert [str(dtype) for dtype in workshop_data.dtypes] == [
    "string", "int64", "int64", "int64"
]
assert workshop_data["participant_id"].is_unique
assert workshop_data.notna().all().all()

OUTPUT_DIR = DEMO_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
demo1_output_names = [
    "inference_summary.csv",
    "prediction_intervals.csv",
    "ols_residuals.png",
]
for output_name in demo1_output_names:
    owned_path = OUTPUT_DIR / output_name
    if owned_path.exists():
        owned_path.unlink()

display(workshop_data)

## Define the model vocabulary

**Ordinary least squares (OLS)** chooses coefficients to minimize the sum of squared residuals. A **response** is the outcome being modeled; an **explanatory variable** supplies a model input. The **intercept** is the conditional mean when included explanatory variables equal zero. A **coefficient** is the conditional mean difference associated with a one-unit explanatory-variable difference while the other included variables are held fixed. A **fitted value** is the model's estimated conditional mean for one row. A **residual** is `observed - fitted`; an unobserved **error** is the population-level departure from the conditional mean.

We fit exactly `assessment_score ~ study_hours + prior_score`. We will report the study-hours estimate and interval—not p-value labels, alternate formulas, or model-selection scores.

In [ ]:
import statsmodels.formula.api as smf

ols_result = smf.ols(
    "assessment_score ~ study_hours + prior_score",
    data=workshop_data,
).fit()

expected_parameters = np.array([
    24.596318298320796,
    1.6452439696265557,
    0.6663592593479781,
])
assert ols_result.params.index.tolist() == [
    "Intercept", "study_hours", "prior_score"
]
assert np.allclose(ols_result.params.to_numpy(), expected_parameters)
study_interval = ols_result.conf_int(alpha=0.05).loc["study_hours"]
assert np.isclose(ols_result.bse["study_hours"], 0.23585218765558658)
assert np.allclose(
    study_interval.to_numpy(),
    [1.111709253959844, 2.1787786852932673],
)
assert np.isclose(ols_result.resid.mean(), -1.0658141036401503e-14)

inference_summary = pd.DataFrame({
    "term": pd.Series(["study_hours"], dtype="string"),
    "estimate": pd.Series([ols_result.params["study_hours"]], dtype="float64"),
    "standard_error": pd.Series([ols_result.bse["study_hours"]], dtype="float64"),
    "ci_lower": pd.Series([study_interval.iloc[0]], dtype="float64"),
    "ci_upper": pd.Series([study_interval.iloc[1]], dtype="float64"),
})
inference_path = OUTPUT_DIR / "inference_summary.csv"
inference_summary.to_csv(
    inference_path,
    index=False,
    lineterminator="\n",
    float_format="%.6f",
)
assert len(inference_path.read_bytes()) == 95
assert hashlib.sha256(inference_path.read_bytes()).hexdigest() == (
    "feccc3b50dcd46cd3bfb0c73d246940244823d2d9711da91ca1515d7bd9a1066"
)
display(inference_summary)
print(
    "Holding included prior score fixed, one additional study hour is "
    "associated with an estimated 1.645244-point difference in population "
    "mean assessment score under the stated model and sampling assumptions. "
    "This is not an intervention effect."
)

## Use the residual plot as a warning diagnostic

A **residual plot** places residuals against fitted values to look for warning patterns. Curvature, changing spread, or an isolated residual can warn that the model is inadequate. A quiet plot cannot prove the assumptions. Before running the next cell, predict the sign meaning: `observed - fitted` above zero means the observed score exceeds its fitted value.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fitted_values = ols_result.fittedvalues.to_numpy()
residual_values = ols_result.resid.to_numpy()
assert len(fitted_values) == len(residual_values) == 12

residual_figure, residual_axis = plt.subplots(figsize=(10, 6))
residual_axis.scatter(
    fitted_values,
    residual_values,
    color="#1f4e79",
    edgecolor="white",
    linewidth=0.7,
    s=70,
    zorder=3,
)
residual_axis.axhline(0.0, color="#b22222", linewidth=1.8, zorder=2)
residual_axis.set_title("Residual check: workshop OLS association")
residual_axis.set_xlabel("Fitted assessment score")
residual_axis.set_ylabel("Residual (observed - fitted)")
residual_axis.grid(alpha=0.2, zorder=1)
assert len(residual_axis.collections[0].get_offsets()) == 12
assert np.allclose(residual_axis.lines[0].get_ydata(), [0.0, 0.0])
residual_figure.tight_layout()
residual_plot_path = OUTPUT_DIR / "ols_residuals.png"
residual_figure.savefig(residual_plot_path, dpi=120)
display(residual_figure)
plt.close(residual_figure)

## Distinguish uncertainty targets before calculating them

A **standard error** estimates the repeated-sample variability of an estimator. A **95% confidence-interval procedure** is designed, under its assumptions, to cover its fixed target in 95% of repeated samples; it is not a 95% probability assigned to this already-computed interval.

A **mean-response confidence interval** targets the population conditional mean at supplied explanatory values. An **individual prediction interval** targets one new outcome and includes both mean-estimation uncertainty and individual variation. Before running the cell, predict why the individual interval should be wider. Neither interval licenses extrapolation or a causal claim.

In [ ]:
new_case = pd.DataFrame({
    "case_id": pd.Series(["new-participant"], dtype="string"),
    "study_hours": pd.Series([5.0], dtype="float64"),
    "prior_score": pd.Series([75.0], dtype="float64"),
})
prediction_frame = ols_result.get_prediction(new_case).summary_frame(alpha=0.05)
prediction_intervals = pd.DataFrame({
    "case_id": new_case["case_id"],
    "study_hours": new_case["study_hours"],
    "prior_score": new_case["prior_score"],
    "predicted_mean": prediction_frame["mean"].astype("float64"),
    "mean_ci_lower": prediction_frame["mean_ci_lower"].astype("float64"),
    "mean_ci_upper": prediction_frame["mean_ci_upper"].astype("float64"),
    "individual_pi_lower": prediction_frame["obs_ci_lower"].astype("float64"),
    "individual_pi_upper": prediction_frame["obs_ci_upper"].astype("float64"),
})
expected_interval_values = np.array([
    5.0, 75.0, 82.799483, 82.349511, 83.249455, 81.338179, 84.260786
])
assert np.allclose(
    prediction_intervals.iloc[0, 1:].to_numpy(dtype="float64"),
    expected_interval_values,
    atol=5e-7,
)
mean_width = prediction_intervals.loc[0, "mean_ci_upper"] - prediction_intervals.loc[0, "mean_ci_lower"]
individual_width = prediction_intervals.loc[0, "individual_pi_upper"] - prediction_intervals.loc[0, "individual_pi_lower"]
assert individual_width > mean_width

intervals_path = OUTPUT_DIR / "prediction_intervals.csv"
prediction_intervals.to_csv(
    intervals_path,
    index=False,
    lineterminator="\n",
    float_format="%.6f",
)
assert len(intervals_path.read_bytes()) == 200
assert hashlib.sha256(intervals_path.read_bytes()).hexdigest() == (
    "6bbe7d288b39efc870ead781363dc747223d336b1fe521c59eec724927469663"
)
serialized_intervals = pd.read_csv(
    intervals_path,
    dtype={
        "case_id": "string",
        "study_hours": "float64",
        "prior_score": "float64",
        "predicted_mean": "float64",
        "mean_ci_lower": "float64",
        "mean_ci_upper": "float64",
        "individual_pi_lower": "float64",
        "individual_pi_upper": "float64",
    },
)
assert np.isclose(
    serialized_intervals.loc[0, "mean_ci_upper"]
    - serialized_intervals.loc[0, "mean_ci_lower"],
    0.899944,
)
assert np.isclose(
    serialized_intervals.loc[0, "individual_pi_upper"]
    - serialized_intervals.loc[0, "individual_pi_lower"],
    2.922607,
)
display(serialized_intervals)

In [ ]:
import struct

inference_readback = pd.read_csv(
    inference_path,
    dtype={
        "term": "string",
        "estimate": "float64",
        "standard_error": "float64",
        "ci_lower": "float64",
        "ci_upper": "float64",
    },
)
interval_readback = pd.read_csv(
    intervals_path,
    dtype={
        "case_id": "string",
        "study_hours": "float64",
        "prior_score": "float64",
        "predicted_mean": "float64",
        "mean_ci_lower": "float64",
        "mean_ci_upper": "float64",
        "individual_pi_lower": "float64",
        "individual_pi_upper": "float64",
    },
)
assert [str(dtype) for dtype in inference_readback.dtypes] == [
    "string", "float64", "float64", "float64", "float64"
]
assert [str(dtype) for dtype in interval_readback.dtypes] == [
    "string", "float64", "float64", "float64", "float64", "float64", "float64", "float64"
]
assert inference_readback.to_dict("records") == [{
    "term": "study_hours",
    "estimate": 1.645244,
    "standard_error": 0.235852,
    "ci_lower": 1.111709,
    "ci_upper": 2.178779,
}]
assert interval_readback.shape == (1, 8)

png_bytes = residual_plot_path.read_bytes()
assert png_bytes[:8] == b"\x89PNG\r\n\x1a\n"
assert struct.unpack(">II", png_bytes[16:24]) == (1200, 720)
assert png_bytes[25] in {2, 6}
assert set(path.name for path in OUTPUT_DIR.iterdir() if path.name in demo1_output_names) == set(demo1_output_names)
print("PASS: Demo 1 fresh inference outputs and residual plot verified.")